# Rigid-body interpolation

We have several reference markers. They are presesnt in the initial and final trials. Additionally, we have several target markers which are only present in the initial trial. We need to insert these into the final trial. Assume that all the markeres form the same rigid body in all the trials.

Read input trials from c3d files and write the result to a new c3d file using ezc3d.

**Note:** Something funny happens with c3d format handling. Running the script twice (e.g. using the output file as input for a new instance of the script) yields a c3d file that looks OK in Mokka, but is total garbage in Vicon Nexus. Running the script once yields a c3d file that looks OK in both.

In [7]:
import numpy as np
from scipy.spatial.transform import Rotation
import ezc3d

In [8]:
INIT_TRIAL = '/home/andrey/scratch/extrapolate_marker/stripped/2022_08_16_01.c3d'
FINAL_TRIAL = '/home/andrey/scratch/extrapolate_marker/stripped/2022_08_16_10.c3d'

REF_MARKER_SETS = [['LSH1', 'LSH2', 'LTIB'], ['RSH1', 'RSH2', 'RTIB'], ['RASI', 'LASI', 'PEL']]
TARG_MARKER_SETS =  [['LANK'], ['RANK'], ['RPSI', 'LPSI']]

In [9]:
def align_vecs(ref_vecs_init, targ_vecs_init, ref_vecs_final):
    
    # Convert from ezc3d format to (N, 3)
    ref_vecs_init = ref_vecs_init[:3, :].T
    targ_vecs_init = targ_vecs_init[:3, :].T
    ref_vecs_final = ref_vecs_final[:3, :].T

    # Center the vectors
    ref_vecs_init_cent = ref_vecs_init - np.mean(ref_vecs_init, axis=0)
    targ_vecs_init_cent = targ_vecs_init - np.mean(ref_vecs_init, axis=0)
    ref_vecs_final_cent = ref_vecs_final - np.mean(ref_vecs_final, axis=0)

    rot, rssd = Rotation.align_vectors(ref_vecs_final_cent, ref_vecs_init_cent, return_sensitivity=False)
    targ_vecs_final = rot.apply(targ_vecs_init_cent) + np.mean(ref_vecs_final, axis=0)

    # Convert back to ezc3d format
    targ_vecs_final = np.hstack((targ_vecs_final, np.ones((targ_vecs_final.shape[0], 1)))).T

    return targ_vecs_final

In [10]:
def rigid_body_extrap(c3d_init, c3d_final, ref_markers, targ_markers):
    ## -----------------------------------------------------------------------------------
    # Process initial trial
    #
    all_markers = c3d_init['parameters']['POINT']['LABELS']['value']

    ref_markers_idx = [all_markers.index(m) for m in ref_markers]
    targ_markers_idx = [all_markers.index(m) for m in targ_markers]

    # Find the first valid frame
    data = c3d_init['data']['points'][:, ref_markers_idx + targ_markers_idx, :]
    is_valid = np.all(~np.isnan(data), axis=(0,1))
    valid_idcs = np.where(is_valid)[0]
    assert len(valid_idcs) > 0, 'Cannot find a frame in which all the markers are present'
    valid_idx = valid_idcs[0]

    # Extract the reference and target vectors
    ref_vecs_init = c3d_init['data']['points'][:, ref_markers_idx, valid_idx]
    targ_vecs_init = c3d_init['data']['points'][:, targ_markers_idx, valid_idx]

    ## -----------------------------------------------------------------------------------
    # Process final trial
    #
    all_markers = c3d_final['parameters']['POINT']['LABELS']['value']

    ref_markers_idx = [all_markers.index(m) for m in ref_markers]

    data = c3d_final['data']['points'][:, ref_markers_idx, :]
    is_valid = np.all(~np.isnan(data), axis=(0,1))
    valid_idcs = np.where(is_valid)[0]

    target_data = np.full((4, len(targ_markers), data.shape[2]), np.nan)
    target_data[3, :, :] = 1

    for idx in valid_idcs:
        ref_vecs_final = data[:, :, idx]
        targ_vecs_final = align_vecs(ref_vecs_init, targ_vecs_init, ref_vecs_final)
        target_data[:, :, idx] = targ_vecs_final

    return target_data

## Extrapolate

In [11]:
c3d_init = ezc3d.c3d(INIT_TRIAL)
c3d_final = ezc3d.c3d(FINAL_TRIAL)

target_datas = (rigid_body_extrap(c3d_init, c3d_final, ref_markers, targ_markers) for ref_markers, targ_markers in zip(REF_MARKER_SETS, TARG_MARKER_SETS))

## Inject the locations into the final file and save

In [12]:
# Delete the target markers from the final c3d file
all_markers = c3d_final['parameters']['POINT']['LABELS']['value']
markers_2_del = set(sum(TARG_MARKER_SETS, [])).intersection(set(all_markers))
idx_2_del = [all_markers.index(m) for m in markers_2_del]

c3d_final['data']['points'] = np.delete(c3d_final['data']['points'], idx_2_del, axis=1)
c3d_final['parameters']['POINT']['LABELS']['value'] = [marker for marker in all_markers if not (marker in markers_2_del)]
c3d_final['parameters']['POINT']['USED']['value'][0] -= len(idx_2_del)

# Add the extrapolated target markers
for ref_markers, targ_markers, target_data in zip(REF_MARKER_SETS, TARG_MARKER_SETS, target_datas):
    c3d_final['parameters']['POINT']['LABELS']['value'].extend(targ_markers)
    c3d_final['data']['points'] = np.concatenate([c3d_final['data']['points'], target_data], axis=1)
    c3d_final['parameters']['POINT']['USED']['value'][0] += len(targ_markers)

del c3d_final['data']['meta_points']
c3d_final.write(FINAL_TRIAL[:-4] + '_extrap.c3d')
